# Diagnostic 2 — offline "test step" on real injections

Each injection was scored **once**, at the exact pre-merger window placement (no slide, no jitter), in two ways:

- **foreground** = noise + this signal &rarr; `fg_chirp_mean`, `fg_chirp_sigma`
- **background** = the *same* noise, no signal &rarr; `bg_chirp_mean`, `bg_chirp_sigma`

Plots below: (1) inferred vs true chirp mass, (2) sigma distributions signal vs noise, (3) z-scores, (4) chirp-mass histograms.

Generate the CSV first with `scripts/diag_test_step.py` (needs a GPU).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = "/n/holystore01/LABS/iaifi_lab/Lab/kyoon/aframe_linoss/runs/regression_sv/diag/premerger_59-60s_ft/test_step.csv"
SNR_MIN = 0.0   # raise to focus on the loud, detectable injections (e.g. 15)

df = pd.read_csv(CSV)
df = df[df.snr >= SNR_MIN]
print(f"{len(df)} injections, SNR >= {SNR_MIN}")
df.describe()[["true_chirp_mass", "fg_chirp_mean", "fg_chirp_sigma", "bg_chirp_sigma", "fg_z"]]

## 1. Inferred vs true chirp mass (signal)
Points should hug the diagonal; spread should shrink for louder (brighter) injections.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(df.true_chirp_mass, df.fg_chirp_mean, c=df.snr, cmap="viridis",
                s=14, alpha=0.6, vmax=np.percentile(df.snr, 95))
lim = [df.true_chirp_mass.min(), df.true_chirp_mass.max()]
ax.plot(lim, lim, "k--", lw=1, label="perfect")
ax.set_xlabel("true chirp mass [$M_\\odot$]"); ax.set_ylabel("inferred chirp mass [$M_\\odot$]")
ax.set_title("Inferred vs true (signal)"); ax.legend()
plt.colorbar(sc, label="SNR"); plt.tight_layout(); plt.show()

## 2. Uncertainty: signal vs background
On signal the model should be **confident** (small sigma); on identical noise it should be **uncertain** (large sigma). Separation here is exactly what the detection statistic exploits.

In [ ]:
lo = min(df.fg_chirp_sigma.min(), df.bg_chirp_sigma.min())
hi = max(df.fg_chirp_sigma.quantile(0.99), df.bg_chirp_sigma.quantile(0.99))
bins = np.linspace(lo, hi, 60)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df.fg_chirp_sigma, bins, density=True, alpha=0.5, color="tab:red", label="signal (foreground)")
ax.hist(df.bg_chirp_sigma, bins, density=True, alpha=0.5, color="tab:blue", label="noise (background)")
ax.set_xlabel("predicted $\\sigma$(chirp mass) [$M_\\odot$]"); ax.set_ylabel("density")
ax.set_title("Predicted uncertainty: signal vs noise"); ax.legend(); plt.tight_layout(); plt.show()

## 3. z-score = (inferred &minus; true) / sigma (signal)
If the uncertainty is well-calibrated this is roughly **standard normal** (dashed). Too narrow &rarr; over-cautious; too wide &rarr; over-confident.

In [ ]:
z = df.fg_z.replace([np.inf, -np.inf], np.nan).dropna()
z = z[np.abs(z) < 10]
bins = np.linspace(-6, 6, 60)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(z, bins, density=True, alpha=0.6, color="tab:purple", label=f"z (mean={z.mean():.2f}, std={z.std():.2f})")
xs = np.linspace(-6, 6, 200)
ax.plot(xs, np.exp(-xs**2 / 2) / np.sqrt(2 * np.pi), "k--", label="unit normal")
ax.set_xlabel("z-score"); ax.set_ylabel("density"); ax.set_title("Calibration of the uncertainty")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Chirp-mass histograms: true vs inferred (signal vs noise)
Signal inferences should track the true distribution; noise inferences are whatever the model says when there is no signal.

In [ ]:
lo = min(df.true_chirp_mass.min(), df.fg_chirp_mean.min(), df.bg_chirp_mean.min())
hi = max(df.true_chirp_mass.max(), df.fg_chirp_mean.max(), df.bg_chirp_mean.max())
bins = np.linspace(lo, hi, 60)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df.true_chirp_mass, bins, density=True, histtype="step", lw=2, color="k", label="true")
ax.hist(df.fg_chirp_mean, bins, density=True, alpha=0.5, color="tab:red", label="inferred (signal)")
ax.hist(df.bg_chirp_mean, bins, density=True, alpha=0.5, color="tab:blue", label="inferred (noise)")
ax.set_xlabel("chirp mass [$M_\\odot$]"); ax.set_ylabel("density")
ax.set_title("Chirp mass: true vs inferred"); ax.legend(); plt.tight_layout(); plt.show()